# Dataset Analysis

This notebook loads data and prepares image dataset for analysis.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns
import os
import cv2
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Mount Google Drive to access files
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import files
import zipfile

# Upload zip file
uploaded = files.upload()

# Unzip the dataset
for filename in uploaded.keys():
    with zipfile.ZipFile(filename, 'r') as zip_ref:
        zip_ref.extractall('/content/coffee_dataset')

print("Dataset extracted successfully!")

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2

# Define dataset path
dataset_path = '/content/drive/MyDrive/Coffee_Bean_Dataset'

# List all classes (folders)
classes = os.listdir(dataset_path)
print(f"Found {len(classes)} classes:")
for i, class_name in enumerate(classes):
    print(f"{i+1}. {class_name}")

# Count images in each class
for class_name in classes:
    class_path = os.path.join(dataset_path, class_name)
    num_images = len(os.listdir(class_path))
    print(f"{class_name}: {num_images} images")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
import pandas as pd

class CoffeeBeanDataset(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = file_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        # Load image
        image = Image.open(self.file_paths[idx]).convert('RGB')

        # Apply transforms
        if self.transform:
            image = self.transform(image)

        label = self.labels[idx]

        return image, label

# Prepare file paths and labels
file_paths = []
labels = []
label_to_idx = {class_name: idx for idx, class_name in enumerate(classes)}

for class_name in classes:
    class_path = os.path.join(dataset_path, class_name)
    for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)
        file_paths.append(img_path)
        labels.append(label_to_idx[class_name])

print(f"Total images: {len(file_paths)}")

# Split dataset
X_train, X_temp, y_train, y_temp = train_test_split(
    file_paths, labels, test_size=0.3, random_state=42, stratify=labels
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train: {len(X_train)}")
print(f"Validation: {len(X_val)}")
print(f"Test: {len(X_test)}")

# Define transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

# Create datasets
train_dataset = CoffeeBeanDataset(X_train, y_train, transform=train_transform)
val_dataset = CoffeeBeanDataset(X_val, y_val, transform=val_transform)
test_dataset = CoffeeBeanDataset(X_test, y_test, transform=val_transform)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

print("Data loaders created successfully!")

In [ ]:
# Visualize sample images from each class
def visualize_dataset(dataset_path, classes, samples_per_class=3):
    fig, axes = plt.subplots(len(classes), samples_per_class,
                             figsize=(15, 5*len(classes)))

    for i, class_name in enumerate(classes):
        class_path = os.path.join(dataset_path, class_name)
        images = os.listdir(class_path)[:samples_per_class]

        for j, img_name in enumerate(images):
            img_path = os.path.join(class_path, img_name)
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            axes[i, j].imshow(img)
            axes[i, j].set_title(f"{class_name}\n{img_name}")
            axes[i, j].axis('off')

    plt.tight_layout()
    plt.show()

# Call the function
visualize_dataset(dataset_path, classes)

In [ ]:
# Get detailed dataset statistics
def analyze_dataset(dataset_path, classes):
    stats = []

    for class_name in classes:
        class_path = os.path.join(dataset_path, class_name)
        images = os.listdir(class_path)

        # Get image sizes
        sizes = []
        for img_name in images[:10]:  # Sample first 10 images
            img_path = os.path.join(class_path, img_name)
            img = cv2.imread(img_path)
            if img is not None:
                h, w, c = img.shape
                sizes.append((h, w))

        stats.append({
            'class': class_name,
            'count': len(images),
            'avg_height': np.mean([s[0] for s in sizes]),
            'avg_width': np.mean([s[1] for s in sizes]),
            'min_size': min(sizes) if sizes else None,
            'max_size': max(sizes) if sizes else None
        })

    # Create DataFrame
    df = pd.DataFrame(stats)
    print(df)

    # Plot distribution
    plt.figure(figsize=(12, 6))
    plt.bar(df['class'], df['count'])
    plt.xticks(rotation=45, ha='right')
    plt.title('Class Distribution')
    plt.ylabel('Number of Images')
    plt.tight_layout()
    plt.show()

    return df

# Run analysis
stats_df = analyze_dataset(dataset_path, classes)

In [ ]:
import instaloader
from itertools import islice

# 1. Initialize Instaloader
L = instaloader.Instaloader(
    download_videos=False,      # We only need images for CNNs
    save_metadata=False,        # Skip extra text files
    post_metadata_txt_pattern='' # Don't save captions
)

# 2. Define your target hashtag
HASHTAG = "greencoffeebeans"

# 3. Download the data
# We use 'islice' to limit the download to the first 100 images
print(f"Starting download for #{HASHTAG}...")
hashtag = instaloader.Hashtag.from_name(L.context, HASHTAG)

for post in islice(hashtag.get_posts(), 100):
    L.download_post(post, target=f"dataset/{HASHTAG}")

print("Download complete!")